## FloPy for exporting model data

This notebook covers shapefile, raster, and VTK exporting capabilities within FloPy 

In [ ]:
import flopy
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

shp_ws = Path("./lv_shapefiles")
vtk_ws = Path("./lv_vtk")

shp_ws.mkdir(exist_ok=True)
vtk_ws.mkdir(exist_ok=True)

## Shapefile exporting

In this example we'll explore the different ways to export shapefiles from model input and output data

### We'll be working with a simplified example version of the Lucerne Valley Model
![lv_model.png](./lv_model.png)

Load and run the simulation

In [ ]:
sim_ws = Path("../../data/lvhm_simple")
sim = flopy.mf6.MFSimulation.load(sim_ws=sim_ws)
sim.run_simulation()

Get the model object and a copy of the modelgrid object

In [ ]:
gwf = sim.get_model()
modelgrid = gwf.modelgrid

Check if the modelgrid has been projected in space

In [ ]:
modelgrid

In [ ]:
print(modelgrid.crs)

In [ ]:
# set CRS to UTM ZONE 11 N : epsg, 26911
epsg = 26911
modelgrid.set_coord_info(crs=f"EPSG:{epsg}")
modelgrid

#### Get a geodataframe of the modelgrid using `to_geodataframe()`

In [ ]:
mgdf = modelgrid.to_geodataframe()
mgdf.head()

plot only the active cells in the geodataframe

In [ ]:
tmp = mgdf[mgdf["active"] == 1]
tmp.plot()

Add additional data from the modelgrid to the geodataframe

In [ ]:
mgdf["top"] = modelgrid.top.ravel()

for i in range(modelgrid.nlay):
    mgdf[f"idomain_{i}"] = modelgrid.idomain[i].ravel()
    mgdf[f"botm_{i}"] = modelgrid.botm[i].ravel()

In [ ]:
mgdf.head()

Export only the active extent of the model to shapefile

In [ ]:
# live code


### Exporting data from MODFLOW Package data objects

Data can be exported from the different modflow package data objects (e.g. Array data and list data) using the `to_geodataframe()` method

#### Array data

In [ ]:
hk = gwf.npf.k
hk

In [ ]:
dfhk = hk.to_geodataframe(full_grid=False)
dfhk.head()

In [ ]:
# check the CRS
dfhk.crs

In [ ]:
# export to file
dfhk.to_file(shp_ws / "hk.shp")

#### List data (record type data)

In [ ]:
wel = gwf.wel

In [ ]:
spd = wel.stress_period_data
spd

In [ ]:
weldf = spd.to_geodataframe(kper=18, full_grid=False,)
weldf.head()

In [ ]:
weldf.to_file(shp_ws / "wel_kper_18.shp")

### Exporting data from an entire package

We can also use `to_geodataframe()` to export all compatible data from a package

In [ ]:
dis = gwf.dis
dis

In [ ]:
disdf = dis.to_geodataframe(shorten_attr=True)
disdf.head()

#### tack on additional package data from npf to the dis geodataframe

In [ ]:
dis_npf_df = gwf.npf.to_geodataframe(gdf=disdf, shorten_attr=True)
dis_npf_df

In [ ]:
dis_npf_df.to_file(shp_ws / "dis_npf.shp")

#### Exporting HFB fault information

In [ ]:
hfb = gwf.hfb
hfb

In [ ]:
hfbdf = hfb.to_geodataframe()
hfbdf.head()

In [ ]:
# set crs and write to file
hfbdf.set_crs(epsg=epsg, inplace=True)
hfbdf.to_file(shp_ws / "hfbs.shp")

## Exporting shapefile data for an entire model

In [ ]:
# live code


### Exporting head output to shapefile

In [ ]:
hds = gwf.output.head()
totims = hds.get_times()

In [ ]:
headdf = hds.to_geodataframe(modelgrid=modelgrid, totim=totims[-1])

In [ ]:
headdf = headdf[headdf["active"] != 0]
headdf.head()

In [ ]:
headdf.to_file(shp_ws / "heads_eos.shp")

### Exporting raster data from a model

Flopy has a built in `Raster` class that can be used to rasterize model data and write it to file. Here is an example of how that is done

In [ ]:
from flopy.utils import Raster

top = modelgrid.top
top[modelgrid.idomain[0] == 0] = -999.

# array must be 3d for a raster (band, nrow, ncol)
top = top.reshape((1, modelgrid.nrow, modelgrid.ncol))


Build a raster object using `raster_from_array()` and write to file

In [ ]:
rstr = Raster.raster_from_array(top, modelgrid=modelgrid, nodataval=-999)
rstr.write(str(shp_ws / "model_top.tif"))

## Exporting a 3d representation of the model to VTK

FloPy uses the Visualization Toolkit to export VTK files that can be visulaized with VTK viewers like Paraview. The `Vtk` class drives this functionality. Here's an example of how this is done:

In [ ]:
from flopy.export.vtk import Vtk

build the FloPy Vtk object (creates the VTK geometries)

In [ ]:
vtkobj = Vtk(model=gwf, vertical_exageration=10)

add package data to the object

In [ ]:
# Add DIS
vtkobj.add_package(gwf.dis)

In [ ]:
# Add NPF
vtkobj.add_package(gwf.npf)

In [ ]:
# Add STO
vtkobj.add_package(gwf.sto)

write to file

In [ ]:
vtkobj.write(vtk_ws / "stuff.vtk")

#### Export the whole model to VTK

In [ ]:
vtkobj = Vtk(model=gwf, vertical_exageration=10, pvd=True)
vtkobj.add_model(gwf)
vtkobj.write(vtk_ws / "lv_model.vtk")

### Exporting particle tracks to Shapefile and csv

Particles paths from PRT and/or MODPATH can also be exported to VTK

In [ ]:
sim_ws = Path("../../data/prt-backward/gwf")
prt_ws = Path("../../data/prt-backward/prt")

gwf_out_ws = Path("./models/prt-backward/gwf")
prt_out_ws = Path("./models/prt-backward/prt")

Load the groundwater flow simulation and run it

In [ ]:
sim0 = flopy.mf6.MFSimulation.load(sim_ws=sim_ws)
sim0.set_sim_path(gwf_out_ws)
sim0.write_simulation()
sim0.run_simulation()

Load the PRT simulation and run it 

In [ ]:
sim1 = flopy.mf6.MFSimulation.load(sim_ws=prt_ws)
sim1.set_sim_path(prt_out_ws)
sim1.write_simulation()
sim1.run_simulation()

Get the GWF model object and the PRT pathline output

In [ ]:
gwf = sim0.get_model()

### Export PRT output to ShapeFile

*Note: I found a small bug in FloPy while setting this up, so we're going to create a reusable hack to enable some built in functionality.*

Create a hack for exporting PRT particle track lines to shapefile

In [ ]:
from flopy.utils.particletrackfile import ParticleTrackFile

class PrtTrackFile(ParticleTrackFile):
    def __init__(self, f):
        super().__init__(f)
        self._data = pd.read_csv(f)
        self._data = self._data.rename(columns={"irpt": "particleid", "t": "time"})
        
        self.kijnames = list(self._data)
        self._data = self._data.to_records(index=False)

    def intersect(self, cells, to_recarray):
        return

Now load our prt file with the hack

In [ ]:
trackcsvfile = "prt-backward-prt.trk.csv"
ptf = PrtTrackFile(prt_out_ws / trackcsvfile)

And use Flopy built in functionality to build and export a particle track shapefile

In [ ]:
gdfprt = ptf.to_geodataframe(gwf.modelgrid)
gdfprt.to_file(shp_ws / "prt_pathlines.shp")

*Another way of doing this would be to load the output CSV and create a point geodataframe*

In [ ]:
dfprt = pd.read_csv(prt_out_ws / trackcsvfile)
geom = gpd.points_from_xy(dfprt["x"], dfprt["y"], dfprt["z"])

In [ ]:
gdfprt2 = gpd.GeoDataFrame(dfprt, geometry=geom)
gdfprt2.head()

In [ ]:
gdfprt2.to_file(shp_ws / "prt_pathline_pts.shp")

### Now exporting to VTK

In [ ]:
# load the prt trackfile dataframe
prtdf = pd.read_csv(prt_out_ws / trackcsvfile)

In [ ]:
# FloPy's Vtk class only supports one pathline set at a time, so rebuild for PRT
vtkp = Vtk(model=gwf, binary=False, xml=True, vertical_exageration=10, pvd=True)
vtkp.add_model(gwf)
vtkp.add_pathline_points(prtdf.to_records(index=False))

vtk_ws = Path("./prt_vtk")
vtk_ws.mkdir(exist_ok=True)
vtkp.write(vtk_ws / "prt_stuff.vtk")